> **⚠️ Windows에서 TensorFlow 실행 시 "DLL load failed" 오류가 나면:**  
> 1) [Google Colab](https://colab.research.google.com)에서 노트북을 열어 실행하거나,  
> 2) [Microsoft Visual C++ Redistributable](https://aka.ms/vs/17/release/vc_redist.x64.exe) (최신) 설치 후 재시도해 보세요.

# [LAB12] 딥러닝 > 신경망의 이해 > 2. 퍼셉트론

## 📘 #01. 퍼셉트론 개요

- 인공신경망(딥러닝)의 기원이 되는 알고리즘.
- 하나 이상의 신호를 입력받아 어떠한 계산을 수행한 후 하나의 Output를 출력한다.
- 퍼셉트론은 1 과 0 의 신호만 가질 수 있다. 신호가 흐르면 1, 흐르지 않으면 0 이다.

### 📝 [1] input이 2개인 퍼셉트론

x1 과 x2 는 입력 신호, y는 출력 신호, w1 과 w2 는 가중치를 의미한다. (w : weight)

**x와 가중치 w를 곱한 값을 모두 더하여 하나의 값(y)로 만들어 낸다.**

입력 신호가 뉴런에 보내질 때는 각각 고유한 가중치가 곱해지고 그 값들을 모두 더해서 나온 값(y)이 어떠한 임계값(θ)을 넘을 때만 1로 출력한다.

신경망에서 만들어진 값(y)을 적절한 출력값으로 변환해 주는 함수를 **활성화 함수**라고 한다.

입력 신호와 출력 신호를 담고있는 원은 **노드** 혹은 **뉴런**이라 부른다.

$$y = \left \{ \begin{array}{cc} {0(w_1x_1 + w_2x_2 \leq \theta)}\\{1(w_1x_1 + w_2x_2 > \theta)} \end{array} \right.$$

#### ✏ (1) 논리회로 / (2) 게이트 / (3) 게이트의 종류

## 📘 #02. AND(OR) Gate

### 📝 [1]. 준비작업

#### ✏ 패키지 참조

In [ ]:
!pip install --upgrade tensorflow

In [ ]:
from hossam import *
from pandas import DataFrame
from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD, RMSprop
from tensorflow.keras.losses import mse
from tensorflow.keras.metrics import RootMeanSquaredError
from tqdm.keras import TqdmCallback

#### ✏ 데이터셋 준비

### 📝 [2] 탐색적 데이터 분석

### 📝 [3] 데이터 전처리

In [ ]:
#origin = load_data("logical_and")
origin = load_data("logical_or")
origin

In [ ]:
# 전체 데이터를 학습용과 테스트용으로 모두 사용
x_train = origin.drop('target', axis=1)
y_train = origin['target']
x_test = x_train.copy()
y_test = y_train.copy()
rows, cols = x_train.shape
print(rows, cols)

### 📝 [4] 신경망 모델 적합

#### ✏ 신경망 모델 구축

In [ ]:
model = Sequential()
model.add(Input(shape=(cols,)))
model.add(Dense(1, activation="linear"))
model.compile(
    optimizer="SGD",
    loss="mse",
    metrics=["accuracy", "mae", RootMeanSquaredError(name="rmse")],
)
model.summary()

#### ✏ 학습하기

In [ ]:
%%time
result = model.fit(
    x_train, y_train,
    epochs=500,
    validation_split=0.2,
    verbose=0,
    callbacks=[TqdmCallback(verbose=1)]
)
result

### 📝 [5] 성능평가

#### ✏ (1) 가중치, 편향 확인

#### ✏ (2) 성능평가 지표

In [ ]:
weight, bias = model.get_weights()
print("가중치: %s" % weight)
print("편향: %s" % bias)

In [ ]:
train_eval = model.evaluate(x_train, y_train, verbose=0, return_dict=True)
test_eval = model.evaluate(x_test, y_test, verbose=0, return_dict=True)
final_results = DataFrame([train_eval, test_eval])
final_results.insert(0, "Dataset", ["Train", "Test"])
final_results["rmse_gap"] = None
final_results.loc[1, "rmse_gap"] = final_results.loc[1, "rmse"] - final_results.loc[0, "rmse"]
final_results

#### ✏ (3) 학습 과정 확인

#### ✏ (4) RMSE 학습곡선

In [ ]:
history_df = DataFrame(data=result.history)
history_df["epoch"] = history_df.index + 1
history_df.head()

In [ ]:
figsize = (1280 / 100, 720 / 100)
fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=100)
sb.lineplot(data=history_df, x="epoch", y="rmse", ax=ax, label="Train RMSE")
sb.lineplot(data=history_df, x="epoch", y="val_rmse", ax=ax, label="Validation RMSE")
history_df["rmse_gap"] = history_df["val_rmse"] - history_df["rmse"]
sb.lineplot(data=history_df, x="epoch", y="rmse_gap", ax=ax, label="RMSE Gap (Val - Train)", linestyle="--")
ax.axhline(0, linestyle=":", linewidth=1)
ax.set_xlabel("Epoch")
ax.set_ylabel("RMSE")
ax.set_title("Training vs Validation RMSE")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
figsize = (1280 / 100, 720 / 100)
fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=100)
sb.lineplot(data=history_df, x="epoch", y="mae", ax=ax, label="Train MAE")
sb.lineplot(data=history_df, x="epoch", y="val_mae", ax=ax, label="Validation MAE")
ax.set_xlabel("Epoch")
ax.set_ylabel("MAE")
ax.set_title("Training vs Validation MAE")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
figsize = (1280 / 100, 720 / 100)
fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=100)
sb.lineplot(data=history_df, x="epoch", y="loss", ax=ax, label="Train Loss")
sb.lineplot(data=history_df, x="epoch", y="val_loss", ax=ax, label="Validation Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.set_title("Training vs Validation Loss")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

### 📝 [6] 학습 결과 적용

#### ✏ (1) 단일 데이터에 대한 예측치 산정

#### ✏ (2) 검증 데이터 전체를 활용하여 예측값 만들기

### 📘 연구과제

logical_xor 데이터셋에 대한 신경망을 구축하고 결과를 확인하라.

In [ ]:
for i in range(0, 2):
    for j in range(0, 2):
        r = model.predict(np.array([[i, j]]), verbose=0)
        print("입력: %d, %d => 출력: %0.2f" % (i, j, r[0][0]))
        print("%s, %s => %s" % (bool(i), bool(j), bool(round(r[0, 0]))))

In [ ]:
r = model.predict(x_test, verbose=0)
r

In [ ]:
origin["pred"] = r.reshape(-1).round()
origin